# 03b Attention network with a refit

In `03-Attention-network.ipynb` the network is fitted on every training year but the last, and the last year is
only used for early stopping. Ridge, in `02-Linear-models.ipynb`, refits on the whole window once $\lambda$ is
chosen, so the network learned from one year less, and the most recent one. This notebook checks whether that
matters, for the main model (attention pooling, Item 7), on both windows. The strict window is the clean test:
there no training label overlaps the test year, so the refit adds data without adding any leak. On the paper
window the last training year's labels run into the test year, so a refit there also brings in that overlap.

For each test year and seed:

1. **No refit** (as in 03): fit on the training years but the last, keep the epoch with the lowest MSE on the last
   year (early stopping, patience 5).
2. **Refit**: start again from the same seed and train on the whole window, last year included, for the number of
   epochs that step 1 chose, with no early stopping. The chunk standardisation and the target's scale are
   recomputed on the whole window. This is the same rule ridge follows: choose the setting on the last year, then
   fit again on everything, with the number of epochs playing the part of $\lambda$.

Everything else is unchanged from 03: the same network and settings (`attention.py`), the same within-year target,
the prediction centred on the test year, so neither version can change the year's level, and scoring with
`scoring.py` against the straight line. Both versions come from the same run, so they are compared on the same
seeds and the same test reports.

**This needs a Colab GPU** (Runtime, Change runtime type, T4). Anywhere else it only builds the model and checks
the split logic. The full run (both windows, 5 seeds, 6 test years) takes about 30 minutes on a T4.

On Colab, the first cell connects Google Drive, gets the repo and copies `data.pkl`, as in 03. Locally it does
nothing.

In [ ]:
import importlib.metadata, importlib.util, os, shutil, subprocess, sys
from pathlib import Path

NOTEBOOK_DIR = "Alex"
IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    # data.pkl was written with pandas 3 and Colab ships pandas 2, which cannot read it. Install pandas 3 before
    # anything imports pandas. If Colab has already loaded pandas, the runtime has to restart: Run all again.
    if int(importlib.metadata.version("pandas").split(".")[0]) < 3:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas==3.0.6"], check=True)
        if "pandas" in sys.modules:
            print("pandas 3 installed but the old one was already loaded: restarting the runtime. Run all again.")
            os.kill(os.getpid(), 9)
    from google.colab import drive
    drive.mount("/content/drive")
    repo = Path("/content/dst-sprint2")
    if repo.exists():
        subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/Oscar339/dst-sprint2.git", str(repo)], check=True)
    processed = repo / "data" / "processed"
    processed.mkdir(parents=True, exist_ok=True)
    if not (processed / "data.pkl").exists():
        shutil.copy(Path("/content/drive/MyDrive/sprint2-data/data.pkl"), processed / "data.pkl")
    os.chdir(repo / NOTEBOOK_DIR)

In [ ]:
import pickle, platform, time
sys.path.insert(0, ".")

import numpy as np
import pandas as pd
import torch
from torch import nn

import scoring
import attention
from attention import HP, DEVICE

print(f"Python {platform.python_version()} | torch {torch.__version__} | device {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else " (checks only; train on a Colab GPU)"))

with open("../data/processed/data.pkl", "rb") as f:   # Oscar's hand-over file (Oscar/make_handover.py)
    d = pickle.load(f)
df = scoring.frame(d)
CHUNKS = {"Item 1": d["X_item1"], "Item 7": d["X_item7"]}        # per report: an (n_chunks x 768) float16 array
x_all, y_all, yr_all = df["logvol_before"].to_numpy(), df["logvol_after"].to_numpy(), df["year"].to_numpy()

LABEL, POOLING = "Item 7", "attention"      # the main model of 03
WINDOWS = ["strict", "paper"]               # strict first: the clean test; paper for comparison with 03
print(f"{len(df):,} reports; settings from attention.py: {HP}")

## 1. Training for a fixed number of epochs

`attention.train_one` (the loop from 03) fits on one set of reports and early-stops on another. The refit needs the
same loop without a validation year: the same optimiser, batches and seed, run for a given number of epochs.

In [ ]:
def train_fixed(chunks, idx, sections, target, stats, seed, n_epochs, hp=HP, pooling="attention"):
    """The loop of attention.train_one, run for exactly n_epochs on idx, with no validation year."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = attention.AttentionNet(hp["width"], hp["attention"], hp["head"], hp["dropout"], pooling,
                                   dim=len(stats[0])).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
    loss_fn = nn.MSELoss()
    t_all = torch.from_numpy(target.astype(np.float32)).to(DEVICE)
    history = []
    for epoch in range(n_epochs):
        model.train()
        t0, total = time.perf_counter(), 0.0
        for b in attention.batches(idx, hp["batch"], rng):
            opt.zero_grad()
            pred, _ = model(*attention.make_batch(chunks, b, sections, stats))
            loss = loss_fn(pred, t_all[b])
            loss.backward()
            opt.step()
            total += loss.item() * len(b)
        history.append({"epoch": epoch, "train_mse": total / len(idx), "seconds": time.perf_counter() - t0})
    return model, pd.DataFrame(history)

## 2. Both versions from one run

`fit_both` trains the 03 model on one split, reads off its best epoch, then refits. It returns both sets of test
predictions and caches them, so `scoring.evaluate` can score each version without training twice. The target is
the same for both: the straight line's residual, with each training year's own average removed. Only the reports
the network learns from, and the statistics computed from them, change.

In [ ]:
FITS, RUN_LOG = {}, []

def fit_both(train, test, seed, label=LABEL, pooling=POOLING, hp=HP):
    key = (tuple(sorted(set(yr_all[train]))), int(yr_all[test][0]), seed)
    if key in FITS:
        return FITS[key]
    sections = attention.sections_of(label)
    years = key[0]
    fit, val = train & (yr_all != years[-1]), train & (yr_all == years[-1])
    fit_idx, val_idx, train_idx, test_idx = (np.flatnonzero(m) for m in (fit, val, train, test))
    assert not (train & test).any() and set(yr_all[train_idx]) == set(years)

    a, b = scoring.fit_line(df, train)                              # the line, on the whole window (as in 03)
    resid = y_all - (a + b * x_all)
    resid = resid - pd.Series(resid).groupby(yr_all).transform("mean").where(train, 0).to_numpy()
    line_test = a + b * x_all[test_idx]

    # 1. No refit, exactly as in 03: fit on all years but the last, early-stop on the last
    t0 = time.perf_counter()
    scale = resid[fit_idx].std()
    stats = attention.chunk_stats(CHUNKS, fit_idx, sections)
    model, hist = attention.train_one(CHUNKS, fit_idx, val_idx, sections, resid / scale, stats, seed, hp, pooling)
    text = attention.predict_text(model, CHUNKS, test_idx, sections, stats, hp) * scale
    best_epoch = int(hist["val_mse"].idxmin())                     # 0-based, so best_epoch + 1 epochs were used
    t1 = time.perf_counter()

    # 2. Refit: the whole window, the same seed, best_epoch + 1 epochs, no early stopping
    scale_r = resid[train_idx].std()
    stats_r = attention.chunk_stats(CHUNKS, train_idx, sections)
    model_r, hist_r = train_fixed(CHUNKS, train_idx, sections, resid / scale_r, stats_r, seed, best_epoch + 1, hp, pooling)
    text_r = attention.predict_text(model_r, CHUNKS, test_idx, sections, stats_r, hp) * scale_r
    t2 = time.perf_counter()

    RUN_LOG.append({"input": label, "pooling": pooling, "seed": seed, "test_year": key[1],
                    "train_years": f"{years[0]}-{years[-1]}", "fit_reports": len(fit_idx),
                    "refit_reports": len(train_idx), "epochs_no_refit": len(hist), "best_epoch": best_epoch,
                    "epochs_refit": len(hist_r), "best_val_mse": float(hist["val_mse"].min() * scale ** 2),
                    "seconds_no_refit": t1 - t0, "seconds_refit": t2 - t1})
    FITS[key] = {"no refit": line_test + text - text.mean(),        # centred on the test year, as in 03
                 "refit": line_test + text_r - text_r.mean()}
    return FITS[key]

def predictor(version, seed):
    return lambda df_, train, test: fit_both(train, test, seed)[version]

## 3. Checks (run anywhere)

Before any training: for every split, the refit reports are the training window with the last year added back, and
none is a test report. Then a two-epoch refit on 64 reports goes through the whole path once, on whatever machine
this is.

In [ ]:
for window in scoring.WINDOWS:
    for T in scoring.TEST_YEARS:
        train, test = scoring.masks(df, T, window)
        years = sorted(set(yr_all[train]))
        fit = train & (yr_all != years[-1])
        assert not (train & test).any()
        assert set(yr_all[train & ~fit]) == {years[-1]}            # the refit adds exactly the last training year
        assert max(years) < T - (1 if window == "strict" else 0)  # strict: nothing filed in the year before T
print("splits OK: the refit adds only the last training year, and never a test report")

sections = attention.sections_of(LABEL)
idx = np.flatnonzero(scoring.masks(df, 2006, "strict")[0])[:64]
stats = attention.chunk_stats(CHUNKS, idx, sections)
target = np.random.default_rng(0).standard_normal(len(df)).astype(np.float32)
m, h = train_fixed(CHUNKS, idx, sections, target, stats, seed=0, n_epochs=2)
p = attention.predict_text(m, CHUNKS, idx, sections, stats)
assert len(h) == 2 and p.shape == (64,) and np.isfinite(p).all()
print(f"two-epoch refit on 64 reports runs: train MSE {h['train_mse'].iloc[0]:.3f} -> {h['train_mse'].iloc[-1]:.3f}")

## 4. The full run (Colab GPU only)

Results are appended to Google Drive after each seed, so if the session dies, running the cell again skips the
seeds already done. At the end both files are copied into `data/output/` in the Colab copy of the repo; download
them from there (or from Drive) and add them to the repo to commit.

In [ ]:
RUN_FULL = IN_COLAB and DEVICE == "cuda"
OUT = Path("/content/drive/MyDrive/sprint2-data/attention") if IN_COLAB else Path("../data/output")
RESULTS_FILE, RUNS_FILE = OUT / "alex-03b-refit-results.csv", OUT / "alex-03b-refit-runs.csv"

if RUN_FULL:
    OUT.mkdir(parents=True, exist_ok=True)
    done = pd.read_csv(RESULTS_FILE)["model"].unique().tolist() if RESULTS_FILE.exists() else []
    t_start = time.perf_counter()
    for seed in HP["seeds"]:
        names = {v: f"{POOLING} pooling, {LABEL}, {v}, seed {seed}" for v in ["no refit", "refit"]}
        if all(n in done for n in names.values()):
            continue
        print(f"seed {seed}   ({(time.perf_counter() - t_start) / 60:.0f} min so far)")
        res = pd.concat([scoring.evaluate(df, predictor(v, seed), name=n, windows=WINDOWS) for v, n in names.items()],
                        ignore_index=True)
        res.to_csv(RESULTS_FILE, mode="a", header=not RESULTS_FILE.exists(), index=False)
        pd.DataFrame(RUN_LOG).to_csv(RUNS_FILE, mode="a", header=not RUNS_FILE.exists(), index=False)
        RUN_LOG.clear()
    for f in [RESULTS_FILE, RUNS_FILE]:
        shutil.copy(f, Path("../data/output") / f.name)
    print(f"done in {(time.perf_counter() - t_start) / 60:.0f} min: "
          f"{pd.read_csv(RESULTS_FILE)['model'].nunique()} version-seed runs, copied to data/output/")
else:
    print("RUN_FULL is False: nothing trained. On a Colab GPU it is True and this cell runs everything.")

## 5. Results

Read from the saved file, so this section runs anywhere once the results are in `data/output/`. For each version:
the change in MSE against the straight line averaged over seeds, per test year and on average, the spread part,
and the typical spread between seeds (the training noise). The no-refit row should match 03's numbers for the same
model up to that noise, since it is the same model retrained; the 03 figure is shown alongside as a check.

In [ ]:
path = Path("../data/output/alex-03b-refit-results.csv")
if path.exists():
    r = pd.read_csv(path)
    r["version"] = r["model"].str.extract(r", (no refit|refit), seed")[0]
    by_year = r.groupby(["window", "version", "test_year"]).agg(change_pct=("change_pct", "mean"),
                                                                 seed_sd=("change_pct", "std")).reset_index()
    display(by_year.pivot_table(index=["window", "version"], columns="test_year", values="change_pct").round(2))

    old = pd.read_csv("../data/output/alex-03-attention-results.csv")
    old = old[old["model"].str.startswith(f"{POOLING} pooling, {LABEL}, seed")]
    summary = (r.groupby(["window", "version"])
                .agg(change_pct=("change_pct", "mean"), spread_change_pct=("spread_change_pct", "mean"),
                     level_shift=("level", lambda s: float((s - r.loc[s.index, "baseline_level"]).abs().max())))
                .join(by_year.groupby(["window", "version"])["seed_sd"].mean().rename("typical_seed_sd")))
    summary["03 (no refit)"] = [old[old.window == w]["change_pct"].mean() for w, _ in summary.index]
    display(summary.round(3))

    runs = pd.read_csv("../data/output/alex-03b-refit-runs.csv")
    print(f"{len(runs)} runs; mean best epoch {runs['best_epoch'].mean() + 1:.1f} of at most {HP['max_epochs']}; "
          f"refit used {runs['refit_reports'].mean() / runs['fit_reports'].mean() - 1:.0%} more reports on average; "
          f"total {(runs['seconds_no_refit'] + runs['seconds_refit']).sum() / 60:.0f} GPU-minutes")
else:
    print("No results yet: run section 4 on a Colab GPU first.")